In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box, Polygon
import fiona, re
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score
from imblearn.over_sampling import RandomOverSampler

DATA COLLECTING

In [2]:
GDB_BANYUMAS = r"E:/KULIAH SEMESTER 7/COMPUTING PROJECT/CAPSTONEPROJECT/2022_RBI25K_KAB_BANYUMAS_KUGI50_20221231.gdb"
GDB_JAKPUS   = r"E:/KULIAH SEMESTER 7/COMPUTING PROJECT/CAPSTONEPROJECT/KotaAdm.JakartaPusat/2022_RBI25K_JAKPUS.gdb"
GDB_YOGYA    = r"E:/KULIAH SEMESTER 7/COMPUTING PROJECT/CAPSTONEPROJECT/KotaYogyakarta/KOTA YOGYAKARTA/2022_RBI25K_KOTA_YOGYAKARTA_KUGI50_20221231.gdb"

# Batas admin (GADM level 3)
banyumas_shp = r"E:/KULIAH SEMESTER 7/COMPUTING PROJECT/CAPSTONEPROJECT/indonesiaSHPLevel3/gadm41_IDN_3.shp"
adm = gpd.read_file(banyumas_shp)

#  AOI: Banyumas
AOI_BANYUMAS = (
    adm[(adm["NAME_2"] == "Banyumas")]
    .dissolve(by="NAME_2")
    .to_crs(4326)
)

#  AOI: Jakarta Pusat
AOI_JAKPUS = (
    adm[(adm["NAME_2"].str.contains("Jakarta Pusat", case=False, na=False))]
    .dissolve(by="NAME_2")
    .to_crs(4326)
)
if AOI_JAKPUS.empty:
    # fallback lain (kalau penamaan beda)
    AOI_JAKPUS = (
        adm[(adm["NAME_1"].str.contains("Jakarta", case=False, na=False)) &
            (adm["NAME_2"].str.contains("Pusat", case=False, na=False))]
        .dissolve(by="NAME_2")
        .to_crs(4326)
    )

#  AOI: Kota Yogyakarta
AOI_YOGYA = (
    adm[(adm["NAME_2"].str.contains("Yogyakarta", case=False, na=False))]
    .dissolve(by="NAME_2")
    .to_crs(4326)
)

In [3]:
# Pilih UTM lokal (South)

def guess_utm_epsg_south(gdf_wgs84):
    centroid = gdf_wgs84.to_crs(4326).geometry.iloc[0].centroid
    lon = centroid.x
    zone = int((lon + 180) // 6) + 1
    return int(f"327{zone:02d}")  # EPSG:327xx (UTM South)

In [4]:
# List & ambil layer dari GDB

def list_layers(gdb_path):
    return fiona.listlayers(gdb_path)

def find_layer(layers, *keywords):
    """
    Cari nama layer yang mengandung semua keyword (case-insensitive).
    Balikin list kandidat.
    """
    pattern = [re.compile(k, re.I) for k in keywords]
    hits = [ly for ly in layers if all(p.search(ly) for p in pattern)]
    return hits

In [5]:
def read_layers_for_features(gdb_path, epsg, 
                             roads_layer=None, buildings_layer=None, green_layer=None):
    layers = list_layers(gdb_path)
    print(f"[INFO] Layers in {Path(gdb_path).name}: {len(layers)} found")

    gdfs = {}
    if roads_layer:
        gdfs["roads"] = gpd.read_file(gdb_path, layer=roads_layer).to_crs(epsg=epsg)
    if buildings_layer:
        gdfs["buildings"] = gpd.read_file(gdb_path, layer=buildings_layer).to_crs(epsg=epsg)
    if green_layer:
        gdfs["green"] = gpd.read_file(gdb_path, layer=green_layer).to_crs(epsg=epsg)
    return gdfs

In [6]:
# Grid 1 km + hitung panjang/luas per sel

def make_grid(aoi_proj, cell_size=1000):
    minx, miny, maxx, maxy = aoi_proj.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    cells = [box(x, y, x+cell_size, y+cell_size) for x in xs for y in ys]
    grid = gpd.GeoDataFrame(geometry=cells, crs=aoi_proj.crs)
    grid = gpd.overlay(grid, aoi_proj, how="intersection")
    grid["km2"] = grid.area / 1e6
    return grid

def length_in_grid(lines_gdf, grid, out_col="road_km"):
    if lines_gdf is None or lines_gdf.empty:
        grid[out_col] = 0.0
        return grid

    # Pastikan line; kalau polygon (kasus jarang), pakai boundary
    sample_geom = lines_gdf.geometry.iloc[0]
    if sample_geom.geom_type not in ["LineString", "MultiLineString"]:
        lines_gdf = gpd.GeoDataFrame(geometry=lines_gdf.boundary, crs=lines_gdf.crs)

    inter = gpd.overlay(lines_gdf[["geometry"]], grid[["geometry"]], how="intersection")
    if inter.empty:
        grid[out_col] = 0.0
        return grid

    inter["len_m"] = inter.length
    sj = gpd.sjoin(grid[["geometry"]], inter[["geometry", "len_m"]], how="left", predicate="intersects")
    sj["len_m"] = sj["len_m"].fillna(0)
    agg = sj.groupby(sj.index)["len_m"].sum()
    grid[out_col] = agg.reindex(grid.index).fillna(0) / 1000.0
    return grid

def area_in_grid(polys_gdf, grid, out_col):
    if polys_gdf is None or polys_gdf.empty:
        grid[out_col] = 0.0
        return grid

    inter = gpd.overlay(polys_gdf[["geometry"]], grid[["geometry"]], how="intersection")
    if inter.empty:
        grid[out_col] = 0.0
        return grid

    inter["a_m2"] = inter.area
    sj = gpd.sjoin(grid[["geometry"]], inter[["geometry", "a_m2"]], how="left", predicate="intersects")
    sj["a_m2"] = sj["a_m2"].fillna(0)
    agg = sj.groupby(sj.index)["a_m2"].sum()
    grid[out_col] = agg.reindex(grid.index).fillna(0)
    return grid

In [7]:
# Filter area hijau
_GREEN_KEYWORDS = [
    "RTH", "HIJAU", "TAMAN", "HUTAN", "KEBUN", "PERKEBUNAN",
    "SABANA", "SEMAK", "RUMPUT", "PADANG RUMPUT",
    "SAWAH", "TEGALAN", "PERKARANGAN", "MANGROVE"
]

def select_green_polygons(landcover_gdf):
    if landcover_gdf is None or landcover_gdf.empty:
        return landcover_gdf

    candidate_cols = [
        "KELAS", "KELAS_LC", "KATEGORI", "JNS_TNH", "JENIS", "FUNGSI", "NAMOBJ",
        "LC_TYPE", "LC_NAME", "USE", "KETERANGAN"
    ]
    cols = [c for c in candidate_cols if c in landcover_gdf.columns]
    if not cols:
        return landcover_gdf.iloc[0:0]

    lc = landcover_gdf.copy()
    lc["_MERGED_"] = lc[cols].astype(str).agg(" ".join, axis=1).str.upper()
    pattern = re.compile("|".join([re.escape(k) for k in _GREEN_KEYWORDS]), re.I)
    green = lc[lc["_MERGED_"].str.contains(pattern, na=False)].drop(columns=["_MERGED_"])
    return green

In [8]:
# Wrapper: AOI + 3 layer → 3 fitur
def features_from_rbi(aoi_wgs84, gdb_path, roads_layer, buildings_layer, green_layer, cell_size=1000):
    epsg = guess_utm_epsg_south(aoi_wgs84)
    aoi = aoi_wgs84.to_crs(epsg)

    gdfs = read_layers_for_features(
        gdb_path, epsg,
        roads_layer=roads_layer,
        buildings_layer=buildings_layer,
        green_layer=green_layer
    )

    roads = gdfs.get("roads", None)
    builds = gdfs.get("buildings", None)
    green  = gdfs.get("green", None)

    if green is not None and not green.empty:
        green = select_green_polygons(green)

    grid = make_grid(aoi, cell_size=cell_size)
    grid = length_in_grid(roads,  grid, out_col="road_km")
    grid = area_in_grid(builds,  grid, out_col="building_m2")
    grid = area_in_grid(green,   grid, out_col="green_m2")

    grid["road_km_per_km2"]   = grid["road_km"] / grid["km2"]
    grid["building_area_pct"] = (grid["building_m2"] / (grid["km2"] * 1e6)) * 100.0
    grid["green_area_pct"]    = (grid["green_m2"]    / (grid["km2"] * 1e6)) * 100.0

    out = grid[[
        "geometry", "km2",
        "road_km_per_km2",
        "building_area_pct",
        "green_area_pct"
    ]].copy()
    out.crs = grid.crs
    return out

In [9]:
# Labeling konsisten (ambang dari Banyumas)
def derive_thresholds_from_banyumas(df_bms, y_bms=None, strategy="quantile"):
    if strategy=="quantile":
        thr = {
            "road_high"    : df_bms["road_km_per_km2"].quantile(0.66),
            "building_high": df_bms["building_area_pct"].quantile(0.66),
            "green_high"   : df_bms["green_area_pct"].quantile(0.66),
            "green_low"    : df_bms["green_area_pct"].quantile(0.33),
        }
        return thr
    else:
        raise NotImplementedError("Tambahkan strategi lain bila perlu.")

def rule_based_label(row, thr):
    r = row["road_km_per_km2"]
    b = row["building_area_pct"]
    g = row["green_area_pct"]

    if (r >= thr["road_high"] and b >= thr["building_high"] and g <= thr["green_low"]):
        return "tinggi"
    elif (g >= thr["green_high"] and r < thr["road_high"] and b < thr["building_high"]):
        return "rendah"
    else:
        return "sedang"

In [10]:
# ISI NAMA LAYER DI SINI (hasil dari list_layers)

LY_ROAD_BMS  = "TRANSPORTASI_JALAN_LN"
LY_BUILD_BMS = "BANGUNAN_AR"
LY_GREEN_BMS = "TUTUPAN_LAHAN_AR"

LY_ROAD_JKT  = "JALAN_LN"
LY_BUILD_JKT = "BANGUNAN_AR"
LY_GREEN_JKT = "TUTUPAN_LAHAN_AR"

LY_ROAD_YGY  = "JALAN_LN"
LY_BUILD_YGY = "BANGUNAN_AR"
LY_GREEN_YGY = "TUTUPAN_LAHAN_AR"


In [12]:
# ============== EKSTRAK FITUR per KOTA (ROBUST TANPA UBAH FUNGSI LAIN) ==============

def pick_first_existing_layer(gdb_path, candidates):
    avail = set(list_layers(gdb_path))
    for name in candidates:
        if name in avail:
            return name
    return None

# Kandidat per tema (urutkan dari yang paling ideal)
ROAD_CANDS  = [
    "JALAN_LN_25K", "TRANSPORTASI_JALAN_LN", "JALAN_LN", "JALAN_LNRS"
]
# Proxy bangunan: pakai layer built-up permukiman/niaga/industri dst kalau BANGUNAN_AR tidak ada
BUILD_CANDS = [
    "BANGUNAN_AR",
    "PERMUKIMAN_AR_25K", "PERUMAHAN_AR_25K",
    "NIAGA_AR_25K", "INDUSTRI_AR_25K", "INDUSTRIPARIWISATA_AR_25K",
    "PENDIDIKAN_AR_25K", "ARENAOLAHRAGA_AR_25K"
]
# Hijau: pilih satu layer hijau yang tersedia (kalau mau gabung banyak layer hijau, nanti bisa kita upgrade)
GREEN_CANDS = [
    "TUTUPAN_LAHAN_AR",  # jika ada
    "HUTANTANAMAN_AR_25K", "HERBADANRUMPUT_AR_25K", "SAWAH_AR_25K",
    "TAMAN_AR_25K", "KEBUN_AR_25K"
]

def resolve_layers_simple(city, gdb_path, ly_road, ly_bldg, ly_green):
    # Jika user sudah isi dan ada, pakai itu; jika tidak, pilih dari kandidat
    r = ly_road  if ly_road  in set(list_layers(gdb_path)) else pick_first_existing_layer(gdb_path, ROAD_CANDS)
    b = ly_bldg  if ly_bldg  in set(list_layers(gdb_path)) else pick_first_existing_layer(gdb_path, BUILD_CANDS)
    g = ly_green if ly_green in set(list_layers(gdb_path)) else pick_first_existing_layer(gdb_path, GREEN_CANDS)

    print(f"[{city}] memakai layer -> road={r}, building={b}, green={g}")
    if not (r and b and g):
        # Cetak daftar layer supaya kamu bisa cek cepat
        print(f"[{city}] Layer masih ada yang None. Daftar layer tersedia:\n{list_layers(gdb_path)}")
        raise ValueError(f"[{city}] Tidak dapat menentukan semua layer (road/building/green). Harap cek nama layer.")
    return r, b, g

print("== Ekstraksi fitur Banyumas ==")
r,b,g = resolve_layers_simple("Banyumas", GDB_BANYUMAS, LY_ROAD_BMS, LY_BUILD_BMS, LY_GREEN_BMS)
bms_feat = features_from_rbi(AOI_BANYUMAS, GDB_BANYUMAS, r, b, g)
bms_feat["city"] = "Banyumas"

print("== Ekstraksi fitur Jakarta Pusat ==")
r,b,g = resolve_layers_simple("Jakarta Pusat", GDB_JAKPUS, LY_ROAD_JKT, LY_BUILD_JKT, LY_GREEN_JKT)
jkt_feat = features_from_rbi(AOI_JAKPUS, GDB_JAKPUS, r, b, g)
jkt_feat["city"] = "JakartaPusat"

print("== Ekstraksi fitur Kota Yogyakarta ==")
r,b,g = resolve_layers_simple("Yogyakarta", GDB_YOGYA, LY_ROAD_YGY, LY_BUILD_YGY, LY_GREEN_YGY)
ygy_feat = features_from_rbi(AOI_YOGYA, GDB_YOGYA, r, b, g)
ygy_feat["city"] = "Yogyakarta"


== Ekstraksi fitur Banyumas ==
[Banyumas] memakai layer -> road=JALAN_LN_25K, building=PERMUKIMAN_AR_25K, green=HUTANTANAMAN_AR_25K
[INFO] Layers in 2022_RBI25K_KAB_BANYUMAS_KUGI50_20221231.gdb: 44 found
== Ekstraksi fitur Jakarta Pusat ==
[Jakarta Pusat] memakai layer -> road=JALAN_LN_25K, building=PERMUKIMAN_AR_25K, green=HERBADANRUMPUT_AR_25K
[INFO] Layers in 2022_RBI25K_JAKPUS.gdb: 61 found


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


== Ekstraksi fitur Kota Yogyakarta ==
[Yogyakarta] memakai layer -> road=JALAN_LN_25K, building=PERMUKIMAN_AR_25K, green=HUTANTANAMAN_AR_25K
[INFO] Layers in 2022_RBI25K_KOTA_YOGYAKARTA_KUGI50_20221231.gdb: 54 found


In [13]:
# Labelkan dengan ambang Banyumas
thr = derive_thresholds_from_banyumas(bms_feat, strategy="quantile")
for df in (bms_feat, jkt_feat, ygy_feat):
    df["label"] = df.apply(lambda r: rule_based_label(r, thr), axis=1)


In [15]:
# Gabungkan & siapkan data ML
# -- Samakan CRS dulu (biar bisa concat)
target_crs = "EPSG:4326"
bms_feat = bms_feat.to_crs(target_crs)
jkt_feat = jkt_feat.to_crs(target_crs)
ygy_feat = ygy_feat.to_crs(target_crs)

all_feat = pd.concat([bms_feat, jkt_feat, ygy_feat], ignore_index=True)

# (opsional) sanity check kolom
req_cols = {"road_km_per_km2","building_area_pct","green_area_pct","label"}
missing = req_cols - set(all_feat.columns)
if missing:
    raise ValueError(f"Kolom wajib hilang: {missing}. Pastikan labeling sudah dibuat sebelum concat.")

X = all_feat[["road_km_per_km2","building_area_pct","green_area_pct"]].copy()
y = all_feat["label"].copy()

# Split stratified
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Balancing khusus TRAIN
ros = RandomOverSampler(random_state=42)
X_tr_bal, y_tr_bal = ros.fit_resample(X_tr, y_tr)


In [19]:
# Pipeline SVM + GridSearchCV
pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("svm", SVC(kernel="rbf", class_weight="balanced", probability=True))
])

param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.1, 0.01, 0.001]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs = GridSearchCV(pipe, param_grid, scoring="accuracy", cv=cv, n_jobs=-1, verbose=1)
gs.fit(X_tr_bal, y_tr_bal)

print("\n[BEST PARAMS]", gs.best_params_)
print("[CV BEST ACC]", gs.best_score_)

y_pred = gs.predict(X_te)
print("\n=== TEST REPORT ===")
print(classification_report(y_te, y_pred))

Fitting 5 folds for each of 16 candidates, totalling 80 fits



[BEST PARAMS] {'svm__C': 100, 'svm__gamma': 'scale'}
[CV BEST ACC] 0.9108631701693479

=== TEST REPORT ===
              precision    recall  f1-score   support

      rendah       0.97      0.96      0.97       162
      sedang       0.75      0.88      0.81        76
      tinggi       0.95      0.82      0.88        90

    accuracy                           0.91       328
   macro avg       0.89      0.89      0.89       328
weighted avg       0.91      0.91      0.91       328

